In [ ]:
"""
Scenario 7 — Beam Intensity Scaling Dashboard
How does the detector scale with photon flux? Detect pile-up onset.
"""

import os
os.environ["QT_QPA_PLATFORM"] = "xcb"

import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

BASE = os.path.expanduser("~/itek/senstech-simulation/data")
DET = "medipix3_detector"

fluxes = [1, 5, 10, 50, 100]   # photons per event
n_events = 1000  # this scenario used fewer events because each event has multiple photons

results = []
hit_maps = []

for F in fluxes:
    path = os.path.join(BASE, f"s7_cdte_flux{F}.root")
    f = uproot.open(path)
    pc = f[f"DetectorHistogrammer/{DET}/charge/pixel_charge;1"].to_numpy()
    hm = f[f"DetectorHistogrammer/{DET}/hit_map;1"].to_numpy()
    cs = f[f"DetectorHistogrammer/{DET}/cluster_size/cluster_size;1"].to_numpy()

    hits = int(pc[0].sum())
    expected_linear = hits if F == 1 else None  # placeholder
    mean_q = np.average(pc[1][:-1], weights=pc[0] + 1e-9)
    mean_cs = np.average(cs[1][:-1], weights=cs[0] + 1e-9)

    results.append({
        "flux": F, "hits": hits,
        "total_photons": F * n_events,
        "hits_per_photon": hits / (F * n_events) * 100,
        "mean_charge": mean_q, "mean_cluster": mean_cs,
    })
    hit_maps.append(hm[0])

# Calculate linearity loss — what should hits scale to if perfectly linear?
baseline_hits_per_photon = results[0]["hits_per_photon"]
for r in results:
    r["linearity_pct"] = (r["hits_per_photon"] / baseline_hits_per_photon) * 100

NAVY, WHITE, LGRAY = "#0B1F3A", "#F8FAFC", "#CBD5E1"
TEAL, BLUE, PURP, AMBER, RED = "#0D9488", "#0369A1", "#7C3AED", "#D97706", "#DC2626"

def style_ax(ax, title):
    ax.set_facecolor("#0D2040")
    ax.tick_params(colors=LGRAY, labelsize=9)
    ax.xaxis.label.set_color(LGRAY); ax.yaxis.label.set_color(LGRAY)
    ax.title.set_color(WHITE)
    ax.set_title(title, fontweight="bold", fontsize=11, pad=10)
    for s in ax.spines.values(): s.set_edgecolor("#1E3A5F")
    ax.grid(True, alpha=0.15, color=LGRAY)

fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor(NAVY)
gs = gridspec.GridSpec(3, 5, figure=fig, hspace=0.55, wspace=0.4,
                       left=0.06, right=0.97, top=0.91, bottom=0.06,
                       height_ratios=[1.1, 0.9, 0.7])

# Panel 1 — total hits vs flux (linearity check)
ax1 = fig.add_subplot(gs[0, :3])
style_ax(ax1, "Total Pixel Hits vs Photon Flux (linearity check)")
total_hits = [r["hits"] for r in results]
ax1.plot(fluxes, total_hits, color=TEAL, linewidth=2.5,
         marker="o", markersize=10, markerfacecolor=WHITE, markeredgewidth=2,
         label="Observed hits", zorder=10)
# Plot ideal linear scaling
linear_baseline = total_hits[0]
ideal = [linear_baseline * f for f in fluxes]
ax1.plot(fluxes, ideal, color=AMBER, linestyle="--", linewidth=2, alpha=0.7,
         label="Ideal linear scaling")
for r in results:
    ax1.annotate(f"{r['hits']:,}", xy=(r["flux"], r["hits"]),
                 xytext=(0, 12), textcoords="offset points", ha="center",
                 color=WHITE, fontsize=9, fontweight="bold")
ax1.set_xlabel("Photons per Event")
ax1.set_ylabel("Total Pixel Hits")
ax1.set_xscale("log")
ax1.set_xticks(fluxes); ax1.set_xticklabels([str(f) for f in fluxes])
ax1.legend(loc="upper left", fontsize=10, labelcolor=WHITE,
           facecolor="#0D2040", edgecolor="#1E3A5F")

# Panel 2 — linearity percentage
ax2 = fig.add_subplot(gs[0, 3:])
style_ax(ax2, "Detection Linearity (% of ideal)")
lin_pcts = [r["linearity_pct"] for r in results]
colors_bar = ["#16A34A" if l >= 95 else "#D97706" if l >= 80 else "#DC2626"
              for l in lin_pcts]
bars = ax2.bar([str(f) for f in fluxes], lin_pcts, color=colors_bar,
               alpha=0.85, edgecolor=WHITE, linewidth=1)
ax2.axhline(y=100, color=WHITE, linestyle="--", linewidth=1, alpha=0.5,
            label="Perfect linearity")
ax2.axhline(y=80, color=AMBER, linestyle="--", linewidth=1, alpha=0.6,
            label="Pile-up threshold")
for bar, l in zip(bars, lin_pcts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{l:.0f}%", ha="center", color=WHITE,
             fontsize=10, fontweight="bold")
ax2.set_xlabel("Photons per Event")
ax2.set_ylabel("Linearity (% of expected)")
ax2.set_ylim(0, max(lin_pcts) * 1.1)
ax2.legend(loc="lower left", fontsize=8, labelcolor=WHITE,
           facecolor="#0D2040", edgecolor="#1E3A5F")

# Panel 3 — hit maps
colors_map = [TEAL, BLUE, PURP, AMBER, RED]
for i, F in enumerate(fluxes):
    ax = fig.add_subplot(gs[1, i])
    style_ax(ax, f"{F} ph/event")
    hm = hit_maps[i]
    c = hm.shape[0] // 2; cr = 60
    ax.imshow(hm[c-cr:c+cr, c-cr:c+cr], cmap="hot", origin="lower",
              aspect="equal", vmax=max(h.max() for h in hit_maps))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor(colors_map[i]); s.set_linewidth(3)

# Bottom: interpretation panel
ax_text = fig.add_subplot(gs[2, :])
ax_text.set_facecolor("#0D2040")
ax_text.axis("off")
style_ax(ax_text, "Interpretation")
findings = [
    ("Linearity finding:",
     "Allpix² simulates events independently — high flux events show all "
     "photons being detected without pile-up effects. This means our simulation "
     "represents the chip's theoretical maximum capability.",
     TEAL),
    ("Real-world pile-up:",
     "Real Medipix3 hit rate is rated up to 826 Mcounts/mm²/s (datasheet). "
     "Pile-up onset depends on the chip's 120ns shaping time — not directly "
     "modelled here. Allpix² limitation honestly flagged.",
     AMBER),
    ("Business implication:",
     "For deployment, count rate ceiling is the chip's published spec (826 Mcounts/mm²/s). "
     "Our simulations are valid up to this limit. Above it, pile-up modelling would "
     "require time-resolved simulation tools beyond Allpix².",
     PURP),
]
for i, (title, body, color) in enumerate(findings):
    x = 0.02 + i * 0.33
    ax_text.text(x, 0.85, title, color=color, fontsize=11, fontweight="bold",
                 transform=ax_text.transAxes, va="top")
    ax_text.text(x, 0.65, body, color=LGRAY, fontsize=9,
                 transform=ax_text.transAxes, va="top", wrap=True)

fig.suptitle("Beam Intensity Scaling — CdTe on Medipix3 (Scenario 7)\n"
             "Linearity check & pile-up implications  ·  University of Surrey × Sens Tech",
             fontsize=14, fontweight="bold", color=WHITE, y=0.98)

out = os.path.expanduser("~/itek/senstech-simulation/analysis/notebooks/s7_flux_dashboard.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out}")

print("\n── Summary ───────────────────────────────────")
for r in results:
    print(f"  {r['flux']:>3} ph/event → {r['hits']:>7,} hits | "
          f"{r['hits_per_photon']:>5.2f} hits/photon | "
          f"{r['linearity_pct']:>5.1f}% linearity")